# 07 — Crow Search Algorithm (CSA) Feature Selection

Binary Crow Search Algorithm (BCSA) is used here because feature selection is a yes/no decision for every feature. The separate continuous CSA demonstration is intentionally omitted because it is not part of the experiment.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Crow Search Algorithm

In [2]:
def sigmoid(x):
    x = np.clip(x, -20, 20)
    return 1.0 / (1.0 + np.exp(-x))


def run_bcsa(
    obj_func,
    n_features,
    pop_size=10,
    iterations=20,
    awareness_probability=0.1,
    flight_length=2.0,
):
    # Each crow position is a binary feature mask.
    positions = np.random.randint(
        0,
        2,
        size=(pop_size, n_features),
    )

    # A valid mask must select at least one feature.
    for position in positions:
        if position.sum() == 0:
            position[np.random.randint(n_features)] = 1

    # Each crow remembers the best position it has personally visited.
    memories = positions.copy()
    memory_scores = np.array([
        obj_func(position)
        for position in memories
    ])

    convergence = []

    for _ in range(iterations):
        candidates = np.empty_like(positions)

        for i in range(pop_size):
            possible = np.delete(np.arange(pop_size), i)
            followed = np.random.choice(possible)

            # If the followed crow is unaware, move toward its memory.
            if np.random.rand() >= awareness_probability:
                r = np.random.rand()
                continuous_position = (
                    positions[i]
                    + r
                    * flight_length
                    * (memories[followed] - positions[i])
                )
            else:
                # If it notices, it deceives the follower into a random move.
                continuous_position = np.random.uniform(
                    -1,
                    1,
                    size=n_features,
                )

            probabilities = sigmoid(continuous_position)
            candidate = (
                np.random.rand(n_features) < probabilities
            ).astype(int)

            if candidate.sum() == 0:
                candidate[np.random.randint(n_features)] = 1

            candidates[i] = candidate

        positions = candidates
        current_scores = np.array([
            obj_func(position)
            for position in positions
        ])

        improved = current_scores < memory_scores
        memories[improved] = positions[improved]
        memory_scores[improved] = current_scores[improved]

        convergence.append(float(memory_scores.min()))

    best_index = np.argmin(memory_scores)

    return (
        memories[best_index].copy(),
        float(memory_scores[best_index]),
        convergence,
    )


## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [3]:
from utils.experiments import run_feature_selector


def csa_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_bcsa(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        iterations=iterations,
        awareness_probability=0.1,
        flight_length=2.0,
    )


csa_results = run_feature_selector("CSA", csa_runner)
csa_results.tail()


Saved: ('breast', 'svm', 0)


Saved: ('breast', 'random_forest', 0)


Saved: ('breast', 'xgboost', 0)


Saved: ('heart', 'svm', 0)


Saved: ('heart', 'random_forest', 0)


Saved: ('heart', 'xgboost', 0)


,Dataset,Classifier,Algorithm,Seed,ValidationFitness,Accuracy,Precision,Recall,F1,ROC_AUC,Features,SelectionRuntime,TestRuntime,SelectedFeatureNames,MaskFile,ConvergenceFile
1,breast,random_forest,CSA,0,0.031053,0.982456,1.000000,0.952381,0.975610,0.998347,15,5.467514,0.265270,"[""radius_mean"", ""perimeter_mean"", ""smoothness_...",results/smoke/artifacts/breast__random_forest_...,results/smoke/artifacts/breast__random_forest_...
2,breast,xgboost,CSA,0,0.032053,0.964912,0.975000,0.928571,0.951220,0.997024,18,2.391949,0.090172,"[""radius_mean"", ""texture_mean"", ""perimeter_mea...",results/smoke/artifacts/breast__xgboost__csa__...,results/smoke/artifacts/breast__xgboost__csa__...
3,heart,svm,CSA,0,0.145891,0.782609,0.822917,0.774510,0.797980,0.873147,15,0.431080,0.020489,"[""age"", ""chol"", ""oldpeak"", ""sex_Female"", ""cp_a...",results/smoke/artifacts/heart__svm__csa__seed0...,results/smoke/artifacts/heart__svm__csa__seed0...
4,heart,random_forest,CSA,0,0.155452,0.798913,0.857143,0.764706,0.808290,0.855691,12,6.690903,0.320773,"[""trestbps"", ""chol"", ""oldpeak"", ""sex_Female"", ...",results/smoke/artifacts/heart__random_forest__...,results/smoke/artifacts/heart__random_forest__...
5,heart,xgboost,CSA,0,0.147491,0.771739,0.812500,0.764706,0.787879,0.882592,19,1.974811,0.084853,"[""age"", ""trestbps"", ""chol"", ""thalch"", ""oldpeak...",results/smoke/artifacts/heart__xgboost__csa__s...,results/smoke/artifacts/heart__xgboost__csa__s...
